# RoadScan — Team v2 controlled training

This notebook follows the project brief and reuses Sara’s original v2 sample: 8,999 training images from Japan, India and Czech Republic, with 1,977 China MotorBike images for validation. It is a clean continuation of the team data work. Keep US and the 40–80 labelled local photos outside training; local photos are final-test only. Previous US results have already been inspected.

## 1. Verify and stage the frozen team sample

The next cell checks the original manifest, image decoding, YOLO boxes, country separation and exact image overlap. It copies the verified sample to Kaggle working storage and saves a portable manifest. No random row split or new sampling is used. Country separation satisfies the project split rule; drive IDs are not available.

In [1]:
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import hashlib, json, math, shutil, platform
import numpy as np
import pandas as pd
from PIL import Image
import yaml
from IPython.display import display, Markdown

BASE = Path('/kaggle/input/notebooks/sarvii7/roadscan-03-sample9000-v2')
SOURCE = BASE/'sample9000'
ROOT = Path('/kaggle/working/team_v2_data')
ART = Path('/kaggle/working/artifacts_team_v2')
ART.mkdir(exist_ok=True)
CLASSES = ['D00','D10','D20','D40']
assert SOURCE.is_dir(), 'Attach the original team sample9000 v2 notebook output.'
original_manifest = (BASE/'sample9000_list.txt').read_text().splitlines()
assert len(original_manifest)==len(set(original_manifest))==8999
jobs=[]
for split,expected in [('train',8999),('val',1977)]:
    images={p.stem:p for p in (SOURCE/split/'images').glob('*.jpg')}
    labels={p.stem:p for p in (SOURCE/split/'labels').glob('*.txt')}
    assert images.keys()==labels.keys() and len(images)==expected, (split,len(images),len(labels))
    if split=='train':
        assert set(images)==set(original_manifest)
        assert Counter(s.split('_')[0] for s in images)=={'Japan':4000,'India':3500,'Czech':1499}
    else:
        assert all(s.startswith('China_MotorBike_') for s in images)
    for folder in ['images','labels']:
        (ROOT/split/folder).mkdir(parents=True,exist_ok=True)
    jobs.extend((split,stem,images[stem],labels[stem]) for stem in sorted(images))

def stage_and_audit(job):
    split,stem,image,label=job
    raw=label.read_text()
    present=[0]*4
    boxes=0
    for line in raw.splitlines():
        if not line.strip():
            continue
        row=list(map(float,line.split()))
        assert len(row)==5 and all(math.isfinite(x) for x in row),str(label)
        c,x,y,w,h=row
        assert c==int(c) and 0<=c<4 and 0<w<=1 and 0<h<=1,str(label)
        assert x-w/2>=-1e-4 and y-h/2>=-1e-4 and x+w/2<=1.0001 and y+h/2<=1.0001,str(label)
        present[int(c)]=1
        boxes+=1
    with Image.open(image) as im:
        im.load()
        width,height=im.size
    digest=hashlib.sha256(image.read_bytes()).hexdigest()
    for src,folder in [(image,'images'),(label,'labels')]:
        shutil.copy2(src,ROOT/split/folder/src.name)
    return dict(split=split,stem=stem,country=stem.split('_')[0],sha256=digest,width=width,height=height,boxes=boxes,empty=int(boxes==0),**dict(zip(CLASSES,present)))

records=[]
with ThreadPoolExecutor(max_workers=8) as pool:
    for i,row in enumerate(pool.map(stage_and_audit,jobs),1):
        records.append(row)
        if i%1000==0:
            print('Audited and staged:',i,'/',len(jobs),flush=True)
manifest=pd.DataFrame(records)
train_hash=set(manifest.loc[manifest.split.eq('train'),'sha256'])
val_hash=set(manifest.loc[manifest.split.eq('val'),'sha256'])
assert not train_hash & val_hash, 'Exact image leakage across train and validation.'
assert set(manifest.loc[manifest.split.eq('train'),'country']).isdisjoint(set(manifest.loc[manifest.split.eq('val'),'country']))
manifest.to_csv(ART/'manifest.csv',index=False)
shutil.copy2(BASE/'sample9000_list.txt',ART/'original_team_sample9000_list.txt')
manifest_digest=hashlib.sha256((chr(10).join(sorted(original_manifest))+chr(10)).encode()).hexdigest()
(ART/'sample_manifest_sha256.txt').write_text(manifest_digest)
DATA=ROOT/'data.yaml'
DATA.write_text(yaml.safe_dump(dict(path=str(ROOT),train='train/images',val='val/images',names=dict(enumerate(CLASSES))),sort_keys=False))
shutil.copy2(DATA,ART/'data.yaml')
summary=manifest.groupby(['split','country']).agg(images=('stem','size'),empty=('empty','sum'),**{c:(c,'sum') for c in CLASSES}).reset_index()
summary.to_csv(ART/'data_summary.csv',index=False)
display(summary)
(ART/'protocol.json').write_text(json.dumps(dict(train=8999,validation=1977,train_countries=['Japan','India','Czech'],validation_country='China_MotorBike',sample_source=str(BASE),sample_sha256=manifest_digest,held_out_country='United_States',local_images='test only; never train or tune',test_evaluated=False,selection='exact original team v2 sample; enriched for D10/D40',split='country-disjoint; exact hashes checked; no verified drive IDs',baseline_comparison='same image assignments; baseline implementation and threshold protocol must be verified before claiming a controlled comparison'),indent=2))
print('DATA_READY',flush=True)
display(Markdown('**Data interpretation.** The exact team v2 sample passed image decoding, label geometry, country separation and exact cross-split duplicate checks. Japan/India/Czech are training countries; China MotorBike is development validation. This preserves the team sample and does not use US or local test images. Sampling enriches rare classes, so training frequencies are not population prevalence.'))


Audited and staged: 1000 / 10976
Audited and staged: 2000 / 10976
Audited and staged: 3000 / 10976
Audited and staged: 4000 / 10976
Audited and staged: 5000 / 10976
Audited and staged: 6000 / 10976
Audited and staged: 7000 / 10976
Audited and staged: 8000 / 10976
Audited and staged: 9000 / 10976
Audited and staged: 10000 / 10976


,split,country,images,empty,D00,D10,D20,D40
0,train,Czech,1499,427,764,300,129,154
1,train,India,3500,500,1012,60,1586,1530
2,train,Japan,4000,500,989,2240,1741,1390
3,val,China,1977,43,1419,738,486,164


DATA_READY


**Data interpretation.** The exact team v2 sample passed image decoding, label geometry, country separation and exact cross-split duplicate checks. Japan/India/Czech are training countries; China MotorBike is development validation. This preserves the team sample and does not use US or local test images. Sampling enriches rare classes, so training frequencies are not population prevalence.

## 2. Train the detector with a longer schedule

Fine-tune COCO-pretrained YOLO11s at 640 pixels on the exact team sample. Allow 80 epochs, patience 15, and a maximum of three hours of training checked at epoch boundaries; preparation and evaluation add time. Keep the previous v2 optimizer and augmentations fixed to test the longer training schedule. Horizontal flips, Mosaic and HSV jitter apply to training only; vertical flips are disabled. China validation selects the checkpoint, so its results are development estimates. Save best/last weights and training history.

Reference: [Ultralytics training documentation](https://docs.ultralytics.com/modes/train/).

In [2]:
import sys, subprocess, importlib.metadata, time
try:
    installed=importlib.metadata.version('ultralytics')
except importlib.metadata.PackageNotFoundError:
    installed=None
if installed!='8.4.177':
    subprocess.check_call([sys.executable,'-m','pip','install','-q','ultralytics==8.4.177'])
import torch, ultralytics
from ultralytics import YOLO
assert DATA.exists() and len(manifest)==10976
assert torch.cuda.is_available(), 'Enable GPU T4 x2 in notebook settings.'
(ART/'environment.json').write_text(json.dumps(dict(python=platform.python_version(),torch=torch.__version__,ultralytics=ultralytics.__version__,gpu=torch.cuda.get_device_name(0)),indent=2))
settings=dict(data=str(DATA),imgsz=640,epochs=80,patience=15,batch=16,device=0,workers=2,seed=42,deterministic=True,optimizer='AdamW',lr0=0.001,lrf=0.01,cos_lr=True,warmup_epochs=3,weight_decay=0.0005,mosaic=1.0,close_mosaic=10,mixup=0.0,scale=0.5,translate=0.1,degrees=0.0,fliplr=0.5,flipud=0.0,hsv_h=0.015,hsv_s=0.7,hsv_v=0.4,cache=False,amp=True,save=True,save_period=10,plots=True,project='/kaggle/working/runs_team_v2',name='yolo11s_640_seed42',exist_ok=False)
(ART/'requested_training_settings.json').write_text(json.dumps(settings,indent=2))
model=YOLO('yolo11s.pt')
clock={}
def start_clock(trainer):
    clock['start']=time.monotonic()
def budget_and_checkpoint(trainer):
    for src,name in [(Path(trainer.best),'best.pt'),(Path(trainer.last),'last.pt'),(Path(trainer.save_dir)/'results.csv','results.csv')]:
        if src.exists():
            shutil.copy2(src,ART/name)
    if time.monotonic()-clock['start']>=180*60:
        trainer.stop=True
        print('Three-hour training budget reached; stopping after this epoch.',flush=True)
model.add_callback('on_train_start',start_clock)
model.add_callback('on_fit_epoch_end',budget_and_checkpoint)
model.train(**settings)
RUN=Path(model.trainer.save_dir)
for src in [RUN/'weights/best.pt',RUN/'weights/last.pt',RUN/'results.csv',RUN/'args.yaml',RUN/'results.png']:
    if src.exists():
        shutil.copy2(src,ART/src.name)
history=pd.read_csv(RUN/'results.csv')
display(history.tail(5))
print('TRAINING_COMPLETE. Epochs:',len(history),flush=True)
display(Markdown('**Training interpretation.** The best checkpoint was selected using China validation detection fitness. The training schedule allows up to 80 epochs with patience 15 and a three-hour epoch-boundary budget. This extends the earlier 40-epoch / 90-minute experiment while keeping the sample, model, resolution, optimizer and augmentations fixed. More training does not guarantee higher accuracy; the evaluation below determines the result.'))


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 546.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 5.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
New https://pypi.org/project/ultralytics/8.4.184 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_pa

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/80      5.03G      2.225      2.693      1.963         30        640: 100% ━━━━━━━━━━━━ 563/563 5.0it/s 1:52
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 62/62 4.0it/s 15.6s
                   all       1977       4650      0.331     0.0823     0.0402      0.011

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/80      5.03G      2.181      2.643      1.942         23        640: 100% ━━━━━━━━━━━━ 563/563 5.0it/s 1:52
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 62/62 3.9it/s 15.8s
                   all       1977       4650      0.114      0.148     0.0479     0.0127

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/80      5.03G      2.136      2.553        1.9         20        640: 100% ━━━━━━━━━━━━ 563/563 5.0it/s 1:53
                 Class     Images  Instances      Box(

,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
59,60,7697.73,1.67992,1.55993,1.52351,0.27660,0.29891,0.21076,0.07832,2.45786,2.69207,2.23733,0.000169,0.000169,0.000169
60,61,7825.45,1.67106,1.52866,1.51218,0.28155,0.28474,0.20043,0.07457,2.47172,2.75492,2.25925,0.000155,0.000155,0.000155
61,62,7953.39,1.65977,1.53713,1.50860,0.27922,0.30040,0.20776,0.07738,2.45033,2.63019,2.23675,0.000142,0.000142,0.000142
62,63,8081.74,1.66356,1.52786,1.50733,0.27505,0.31133,0.20766,0.07765,2.47160,2.70234,2.25288,0.000129,0.000129,0.000129
63,64,8209.42,1.65816,1.52623,1.50724,0.28915,0.30154,0.21154,0.07867,2.47507,2.73980,2.26162,0.000116,0.000116,0.000116


TRAINING_COMPLETE. Epochs: 64


**Training interpretation.** The best checkpoint was selected using China validation detection fitness. The training schedule allows up to 80 epochs with patience 15 and a three-hour epoch-boundary budget. This extends the earlier 40-epoch / 90-minute experiment while keeping the sample, model, resolution, optimizer and augmentations fixed. More training does not guarantee higher accuracy; the evaluation below determines the result.

## 3. Evaluate and interpret the development results

Report detection mAP@50, mAP@50–95, per-class AP, precision/recall, confusion matrices and confidence curves. For comparison with the image classifier, convert detections to image-level scores and report Macro-F1 separately. Save both fixed 0.5 and validation-selected image thresholds; the tuned result is optimistic because the same China validation set selected the checkpoint. Do not evaluate or tune on US/local photos in this training run. Measure a small CPU latency sample and save all artifacts on Kaggle. The cell generates an English interpretation from the actual results.

In [3]:
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
best=YOLO(str(ART/'best.pt'))
det=best.val(data=str(DATA),split='val',imgsz=640,batch=16,device=0,workers=2,conf=0.001,iou=0.7,max_det=300,plots=True,project='/kaggle/working/evaluation_team_v2',name='china_development')
detection_metrics={str(k):float(v) for k,v in det.results_dict.items()}
(ART/'china_detection.json').write_text(json.dumps(detection_metrics,indent=2))
per_class=pd.DataFrame(dict(cls=CLASSES,AP50=[float(x) for x in det.box.ap50],AP50_95=[float(x) for x in det.box.ap]))
per_class.to_csv(ART/'china_per_class_detection.csv',index=False)
display(per_class)
part=manifest.loc[manifest.split.eq('val')].sort_values('stem')
stems=part.stem.tolist()
lookup={s:i for i,s in enumerate(stems)}
truth=part[CLASSES].to_numpy(dtype=int)
scores=np.zeros((len(stems),4),dtype=float)
for result in best.predict(source=str(ROOT/'val/images'),imgsz=640,conf=0.001,iou=0.7,max_det=300,device=0,stream=True,verbose=False):
    i=lookup[Path(result.path).stem]
    for c,p in zip(result.boxes.cls.cpu().numpy().astype(int),result.boxes.conf.cpu().numpy()):
        scores[i,c]=max(scores[i,c],float(p))
grid=np.arange(0.05,0.96,0.05)
thresholds=[]
for c in range(4):
    fs=[precision_recall_fscore_support(truth[:,c],scores[:,c]>=t,average='binary',zero_division=0)[2] for t in grid]
    thresholds.append(float(grid[int(np.argmax(fs))]))
rows=[]
macro={}
for mode,limits in [('fixed_0.5',np.full(4,0.5)),('validation_tuned_optimistic',np.array(thresholds))]:
    p,r,f,support=precision_recall_fscore_support(truth,scores>=limits,zero_division=0)
    macro[mode]=float(f.mean())
    for c,name in enumerate(CLASSES):
        prevalence=float(truth[:,c].mean())
        rows.append(dict(cls=name,mode=mode,threshold=float(limits[c]),precision=float(p[c]),recall=float(r[c]),f1=float(f[c]),support=int(support[c]),always_positive_f1=2*prevalence/(1+prevalence),auroc=float(roc_auc_score(truth[:,c],scores[:,c])) if len(np.unique(truth[:,c]))==2 else None))
image_metrics=pd.DataFrame(rows)
image_metrics.to_csv(ART/'china_image_metrics.csv',index=False)
np.savez_compressed(ART/'china_image_scores.npz',stems=np.array(stems),truth=truth,scores=scores)
display(image_metrics)
config=dict(classes=CLASSES,imgsz=640,conf_floor=0.001,iou=0.7,max_det=300,image_thresholds=thresholds,image_threshold_source='China validation; optimistic development selection',box_confidence=0.25,box_confidence_status='provisional; review validation precision/recall and F1-confidence curves before deployment')
(ART/'inference_config.json').write_text(json.dumps(config,indent=2))
# CPU latency includes model prediction and preprocessing, not app upload/retrieval.
cpu=YOLO(str(ART/'best.pt'))
torch.set_num_threads(2)
latency_paths=[ROOT/'val/images'/(s+'.jpg') for s in stems[:20]]
for _ in range(3):
    cpu.predict(str(latency_paths[0]),imgsz=640,conf=0.25,device='cpu',verbose=False)
latencies=[]
for image in latency_paths:
    start=time.perf_counter()
    cpu.predict(str(image),imgsz=640,conf=0.25,device='cpu',verbose=False)
    latencies.append(time.perf_counter()-start)
pd.DataFrame(dict(image=[p.name for p in latency_paths],seconds=latencies)).to_csv(ART/'cpu_latency.csv',index=False)
latency=dict(images=len(latencies),warmup=3,torch_threads=2,median_seconds=float(np.median(latencies)),p95_seconds=float(np.percentile(latencies,95)),scope='detector prediction only; not complete application latency')
summary=dict(status='complete',epochs_completed=len(history),train_images=8999,validation_images=1977,model='YOLO11s',imgsz=640,detection=detection_metrics,image_macro_f1=macro,cpu_latency=latency,US_evaluated=False,local_evaluated=False,sample_sha256=manifest_digest,baseline_comparison='team v2 image assignments preserved; verify baseline implementation/threshold protocol before claiming controlled E1',pending_experiments=['E1 verified baseline comparison','E2 nano/small and 640/960','E3 augmentation ablation','E4 frozen-model US/local evaluation','E5 retrieval comparison owned by team'])
(ART/'summary.json').write_text(json.dumps(summary,indent=2))
weak=image_metrics.loc[image_metrics['mode'].eq('fixed_0.5')].sort_values('f1').iloc[0]
interpretation=(f"### Development results — interpretation\n\n"
    f"China detection mAP@50 is **{detection_metrics['metrics/mAP50(B)']:.2%}** and mAP@50–95 is **{detection_metrics['metrics/mAP50-95(B)']:.2%}**. "
    f"Image-level Macro-F1 is **{macro['fixed_0.5']:.2%}** at 0.5, or **{macro['validation_tuned_optimistic']:.2%}** after validation threshold tuning. "
    f"These answer different questions: detection evaluates boxes; image F1 evaluates damage presence. "
    f"The weakest image-level class at 0.5 is **{weak['cls']}** (F1 {weak['f1']:.2%}, recall {weak['recall']:.2%}).\n\n"
    f"Median detector CPU latency is **{latency['median_seconds']:.3f} s/image** for 20 images after warm-up, using two CPU threads. This excludes the rest of the app.\n\n"
    "China selected the checkpoint and tuned image thresholds, so these are development results, not proof of local-road generalization. "
    "Compare against the baseline only after verifying matching image assignments and evaluation rules. Review missed potholes and false alarms before selecting the operational box threshold. "
    "Freeze the final recipe before evaluating US/local photos. Local test images must never be used for training or hyperparameter selection.")
display(Markdown(interpretation))
(ART/'RESULTS_INTERPRETATION.md').write_text(interpretation)
print('ALL_COMPLETE',json.dumps(summary),flush=True)


Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11s summary (fused): 100 layers, 9,414,348 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2266.6±519.4 MB/s, size: 79.0 KB)
val: Scanning /kaggle/working/team_v2_data/val/labels.cache... 1977 images, 43 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1977/1977 592.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 124/124 4.0it/s 31.0s
                   all       1977       4650      0.298      0.304      0.227     0.0873
                   D00       1419       2678      0.323      0.152      0.157     0.0561
                   D10        738       1096      0.315      0.335      0.256     0.0723
                   D20        486        641      0.436      0.466      0.413      0.201
                   D40        164        235      0.117      0.264     0.0824     0.0195
Speed: 0.9ms preprocess, 1

,cls,AP50,AP50_95
0,D00,0.157423,0.056113
1,D10,0.255549,0.072332
2,D20,0.413010,0.201363
3,D40,0.082399,0.019490


,cls,mode,threshold,precision,recall,f1,support,always_positive_f1,auroc
0,D00,fixed_0.5,0.50,1.000000,0.031008,0.060150,1419,0.835689,0.893760
1,D10,fixed_0.5,0.50,0.968254,0.082656,0.152310,738,0.543646,0.944368
2,D20,fixed_0.5,0.50,1.000000,0.257202,0.409165,486,0.394641,0.915047
3,D40,fixed_0.5,0.50,0.530120,0.268293,0.356275,164,0.153199,0.842735
4,D00,validation_tuned_optimistic,0.05,0.933891,0.786469,0.853864,1419,0.835689,0.893760
5,D10,validation_tuned_optimistic,0.10,0.802410,0.902439,0.849490,738,0.543646,0.944368
6,D20,validation_tuned_optimistic,0.15,0.826790,0.736626,0.779108,486,0.394641,0.915047
7,D40,validation_tuned_optimistic,0.35,0.393258,0.426829,0.409357,164,0.153199,0.842735


### Development results — interpretation

China detection mAP@50 is **22.71%** and mAP@50–95 is **8.73%**. Image-level Macro-F1 is **24.45%** at 0.5, or **72.30%** after validation threshold tuning. These answer different questions: detection evaluates boxes; image F1 evaluates damage presence. The weakest image-level class at 0.5 is **D00** (F1 6.02%, recall 3.10%).

Median detector CPU latency is **0.164 s/image** for 20 images after warm-up, using two CPU threads. This excludes the rest of the app.

China selected the checkpoint and tuned image thresholds, so these are development results, not proof of local-road generalization. Compare against the baseline only after verifying matching image assignments and evaluation rules. Review missed potholes and false alarms before selecting the operational box threshold. Freeze the final recipe before evaluating US/local photos. Local test images must never be used for training or hyperparameter selection.

ALL_COMPLETE {"status": "complete", "epochs_completed": 64, "train_images": 8999, "validation_images": 1977, "model": "YOLO11s", "imgsz": 640, "detection": {"metrics/precision(B)": 0.2978121059947873, "metrics/recall(B)": 0.30414246734780614, "metrics/mAP50(B)": 0.22709511612951033, "metrics/mAP50-95(B)": 0.08732461846272412, "fitness": 0.08732461846272412}, "image_macro_f1": {"fixed_0.5": 0.24447514883741717, "validation_tuned_optimistic": 0.7229545142764884}, "cpu_latency": {"images": 20, "warmup": 3, "torch_threads": 2, "median_seconds": 0.16377357999954256, "p95_seconds": 0.16858919154983595, "scope": "detector prediction only; not complete application latency"}, "US_evaluated": false, "local_evaluated": false, "sample_sha256": "d88d9d98c1e9e81f21930b6f97fa49c6ceec218938834b9ace757a24baedaffc", "baseline_comparison": "team v2 image assignments preserved; verify baseline implementation/threshold protocol before claiming controlled E1", "pending_experiments": ["E1 verified baseline c